# Variables et préparation des données

Pour éviter toute fuite, une information n'entre dans le modèle que si elle est connue à la seconde où le client valide son panier. Elle est appliquée en trois couches, chacune vérifiée par des tests
automatisés :
1. une liste blanche de colonnes brutes (`COLONNES_BRUTES`) : le modèle ne lit rien d'autre. Ce
   sont aussi les informations que l'API recevra pour chaque commande ;
2. des variables calculées commande par commande (`construire_variables`) : le résultat est le
   même pour une commande seule dans l'API et pour 88 000 commandes à l'entraînement
3. un encodage appris sur l'entraînement seulement (`preparation`) : les informations qui
   dépendent d'autres commandes (le délai habituel d'un vendeur par exemple) sont apprises sur les
   commandes déjà livrées, jamais sur la commande à prédire.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

from olist_promesse.donnees import charger_commandes, selectionner_commandes
from olist_promesse.evaluation import CIBLE, MOIS_VALIDATION, decoupage_mensuel
from olist_promesse.modeles import preparation
from olist_promesse.variables import COLONNES_BRUTES, VARIABLES_CATEGORIELLES, construire_variables

commandes, _ = selectionner_commandes(charger_commandes())

## 1. Les colonnes lues par le modèle

In [2]:
exclues = [colonne for colonne in commandes.columns if colonne not in COLONNES_BRUTES]
print(f"{len(COLONNES_BRUTES)} colonnes lues par le modèle, {len(exclues)} exclues :")
exclues

16 colonnes lues par le modèle, 11 exclues :


['order_id',
 'delai_jours',
 'promesse_jours',
 'n_vendeurs',
 'n_paiements',
 'seller_zip_code_prefix',
 'customer_zip_code_prefix',
 'apres_achat_validation',
 'apres_achat_remise_transporteur',
 'apres_achat_delai_vendeur_jours',
 'apres_achat_note_avis']

Sur les 27 colonnes de l'extraction, 16 sont lues par le modèle. Les 11 autres sont exclues :

| Colonne exclue | Raison |
|---|---|
| `delai_jours` | la cible |
| `apres_achat_validation`, `apres_achat_remise_transporteur`, `apres_achat_note_avis` | connues seulement après l'achat |
| `apres_achat_delai_vendeur_jours` | calculée à partir de la validation du paiement, donc après l'achat (section 2) |
| `promesse_jours` | connue à l'achat, mais c'est la référence à battre (la donner au modèle le ferait dépendre du système qu'il doit remplacer) |
| `order_id` | identifiant, sans information |
| `n_vendeurs` | toujours égal à 1 après la sélection des commandes |
| `n_paiements` | sans lien avec le délai |
| `seller_zip_code_prefix`, `customer_zip_code_prefix` | plus de 14 000 valeurs pour les clients (remplacés par la distance et les États) |

La date d'achat est lue, mais le modèle n'en garde que le jour de la semaine (section 3).

## 2. Une colonne suspecte : le délai accordé au vendeur

`apres_achat_delai_vendeur_jours` est le nombre de jours entre la validation du paiement et la date
limite d'expédition imposée au vendeur. Si c'était un réglage propre à chaque vendeur, il serait connu
à l'achat et ferait une bonne variable. On vérifie s'il dépend plutôt du jour de la validation.

In [3]:
jours = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
delai_vendeur = commandes["apres_achat_delai_vendeur_jours"]
jour_validation = commandes["apres_achat_validation"].dt.day_of_week.map(dict(enumerate(jours)))
(
    pd.crosstab(jour_validation, delai_vendeur.where(delai_vendeur.between(2, 9)), normalize="index")
    .reindex(jours)
    .mul(100)
    .round(0)
    .rename_axis(index="jour de validation du paiement", columns="jours accordés au vendeur (%)")
)

jours accordés au vendeur (%),2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0
jour de validation du paiement,,,,,,,,
lundi,8.0,3.0,59.0,0.0,10.0,11.0,7.0,4.0
mardi,8.0,4.0,2.0,1.0,65.0,12.0,6.0,3.0
mercredi,6.0,0.0,2.0,3.0,66.0,14.0,6.0,3.0
jeudi,0.0,1.0,9.0,3.0,69.0,12.0,6.0,0.0
vendredi,0.0,1.0,8.0,10.0,66.0,14.0,0.0,1.0
samedi,1.0,5.0,10.0,64.0,13.0,0.0,1.0,6.0
dimanche,5.0,2.0,68.0,11.0,0.0,2.0,7.0,4.0


Le délai accordé au vendeur dépend du jour de la validation. 4 jours le plus souvent pour une
validation le lundi ou le dimanche (59% et 68% des cas respectivement), 6 jours du mardi au vendredi (2 cas sur 3 environ), 5 jours le samedi (64% des cas). On peut penser que c'est plutôt une règle
de jours ouvrés appliquée à la date de validation et non pas un réglage du vendeur.

Or la validation a lieu après l'achat, jusqu'à 29 heures plus tard pour un boleto. La colonne est
donc exclue. L'essentiel de son information, l'effet du calendrier, est capté légitimement par le
jour de la semaine de l'achat.

## 3. Les variables calculées

In [4]:
variables = construire_variables(commandes)
pd.DataFrame({
    "type": variables.dtypes.astype(str),
    "manquants (%)": 100 * variables.isna().mean(),
    "valeurs distinctes": variables.nunique(),
}).round(2)

,type,manquants (%),valeurs distinctes
distance_km,float64,0.50,83214
meme_etat,int64,0.00,2
jour_semaine_achat,int32,0.00,7
prix_total,float64,0.00,6754
fret_total,float64,0.00,7323
poids_total_g,float64,0.02,2621
volume_total_cm3,float64,0.02,5212
n_articles,int64,0.00,17
n_echeances,float64,0.00,24
customer_state,str,0.00,27


Le modèle reçoit 14 variables :
- numériques : la distance vendeur-client, l'indicateur « même État », le jour de la semaine de
  l'achat, le prix, les frais de port, le poids, le volume, le nombre d'articles et le nombre
  d'échéances de paiement
- catégorielles : les États du client et du vendeur, le moyen de paiement et la catégorie du
  produit principal
- le vendeur, remplacé par son délai habituel (section 4).

Les valeurs manquantes sont rares (0,5 % des distances, 1,5 % des catégories) et laissées telles
quelles.

Deux informations tirées de la date d'achat ont été écartées :

In [5]:
achat = commandes["order_purchase_timestamp"]
tranche_horaire = pd.cut(achat.dt.hour, [-1, 5, 11, 17, 23], labels=["0 h – 5 h", "6 h – 11 h", "12 h – 17 h", "18 h – 23 h"])
display(commandes.groupby(tranche_horaire, observed=True)[CIBLE].agg(["size", "median"]).round(2))
achat.dt.year.groupby(achat.dt.month).nunique().rename("années observées pour ce mois calendaire").to_frame().T

,size,median
order_purchase_timestamp,,
0 h – 5 h,4215,10.89
6 h – 11 h,19876,10.44
12 h – 17 h,34117,10.81
18 h – 23 h,30484,10.75


order_purchase_timestamp,1,2,3,4,5,6,7,8,9,10,11,12
années observées pour ce mois calendaire,2,2,2,2,2,2,2,1,1,1,1,1


- l'heure d'achat : le délai médian ne varie qu'entre 10,4 et 10,9 jours selon le moment de la
  journée ;
- le mois d'achat : chaque mois calendaire n'est observé qu'une ou deux fois. Le modèle ne
  pourrait pas distinguer un effet saisonnier d'un événement ponctuel et apprendrait ces accidents comme s'ils allaient se répéter.

## 4. L'historique du vendeur : un encodage calculé hors pli

Le vendeur est une information précieuse (exploration, section 6), mais il y en a près de 3 000. On
le remplace alors par son **délai moyen sur les commandes d'entraînement**. C'est ce qu'on appelle un encodage par la cible.

Calculée naïvement, la moyenne d'un vendeur inclut le délai de la commande elle-même.
Pour un vendeur qui n'a qu'une commande d'entraînement, la variable *est* la cible. Le modèle apprend
alors à trop faire confiance à cette variable, et se trompe sur les commandes futures, dont il ne
connaît évidemment pas le délai.

Pour éviter ce piège, `TargetEncoder` de scikit-learn calcule l'encodage hors pli pour les commandes
d'entraînement. Elles sont réparties en 5 plis, et la moyenne de chaque commande est calculée sur les
4 autres. Il lisse aussi la moyenne des vendeurs peu actifs vers la moyenne générale. Pour les
commandes à prédire, il utilise la moyenne de tout l'entraînement, et la moyenne générale pour un
vendeur inédit.

On compare les deux encodages sur le découpage d'avril 2018.

In [6]:
entrainement, evaluation = decoupage_mensuel(commandes, pd.Period("2018-04", freq="M"))
moyennes = entrainement.groupby("seller_id")[CIBLE].mean()
prep = preparation()
X_entrainement = prep.fit_transform(entrainement, entrainement[CIBLE])
X_evaluation = prep.transform(evaluation)

print(f"Vendeurs n'ayant qu'une commande d'entraînement : {(entrainement['seller_id'].value_counts() == 1).sum()} "
      f"sur {entrainement['seller_id'].nunique()}")
pd.DataFrame({
    "naïf (moyenne incluant la commande)": [
        entrainement["seller_id"].map(moyennes).corr(entrainement[CIBLE], method="spearman"),
        evaluation["seller_id"].map(moyennes).fillna(entrainement[CIBLE].mean()).corr(evaluation[CIBLE], method="spearman"),
    ],
    "hors pli (TargetEncoder)": [
        X_entrainement["seller_id"].corr(entrainement[CIBLE], method="spearman"),
        X_evaluation["seller_id"].corr(evaluation[CIBLE], method="spearman"),
    ],
}, index=["entraînement", "avril 2018"]).round(3).rename_axis("corrélation de Spearman avec le délai")

Vendeurs n'ayant qu'une commande d'entraînement : 374 sur 1991


,naïf (moyenne incluant la commande),hors pli (TargetEncoder)
corrélation de Spearman avec le délai,,
entraînement,0.358,0.290
avril 2018,0.237,0.238


L'encodage naïf paraît bien plus lié au délai sur l'entraînement (0,36) que sur les commandes d'avril
(0,24). L'écart vient de la commande elle-même, incluse dans sa propre moyenne. L'encodage hors pli
donne une image plus honnête (0,29 contre 0,24). L'écart restant vient de la dérive, avril ne
ressemblant pas exactement au passé.

Reste à vérifier l'effet sur un modèle. On entraîne le même gradient boosting avec chacun des deux
encodages, sur les 6 mois de validation. Le modèle sert ici d'instrument de mesure seulement (aucun réglage).

In [7]:
def mae_selon_encodage(mois):
    """MAE d'un même modèle entraîné avec l'encodage hors pli du vendeur, puis avec l'encodage naïf."""
    entrainement, evaluation = decoupage_mensuel(commandes, mois)
    prep = preparation()
    hors_pli = (prep.fit_transform(entrainement, entrainement[CIBLE]), prep.transform(evaluation))
    moyennes = entrainement.groupby("seller_id")[CIBLE].mean()
    naif = (
        hors_pli[0].assign(seller_id=entrainement["seller_id"].map(moyennes)),
        hors_pli[1].assign(seller_id=evaluation["seller_id"].map(moyennes).fillna(entrainement[CIBLE].mean())),
    )
    resultats = {}
    for nom, (X_entrainement, X_evaluation) in {"hors pli": hors_pli, "naïf": naif}.items():
        modele = HistGradientBoostingRegressor(
            loss="absolute_error", categorical_features=VARIABLES_CATEGORIELLES, random_state=0
        ).fit(X_entrainement, entrainement[CIBLE])
        resultats[f"{nom}, entraînement"] = mean_absolute_error(entrainement[CIBLE], modele.predict(X_entrainement))
        resultats[f"{nom}, mois évalué"] = mean_absolute_error(evaluation[CIBLE], modele.predict(X_evaluation))
    return pd.Series(resultats)


maes = pd.DataFrame({str(mois): mae_selon_encodage(mois) for mois in MOIS_VALIDATION}).T
maes.loc["moyenne"] = maes.mean()
maes.round(2)

,"hors pli, entraînement","hors pli, mois évalué","naïf, entraînement","naïf, mois évalué"
2017-11,3.67,6.26,3.55,6.32
2017-12,3.63,6.59,3.59,6.65
2018-01,3.93,5.55,3.83,5.60
2018-02,4.23,7.28,4.14,7.33
2018-03,4.33,6.98,4.21,7.00
2018-04,4.47,4.15,4.43,4.24
moyenne,4.04,6.13,3.96,6.19


Le résultat est le même chaque mois : avec l'encodage naïf, le modèle fait mieux sur l'entraînement
(MAE moyenne de 3,96 jours contre 4,04) mais moins bien sur le mois évalué (6,19 jours contre
6,13). L'encodage naïf flatte le modèle là où on connaît déjà la réponse, et le dégrade là où il doit
prévoir. L'écart est modeste, parce que la plupart des vendeurs ont assez de commandes pour que la
leur pèse peu dans leur moyenne, mais il est systématique.

On remarque aussi l'écart entre l'entraînement (environ 4 jours) et les mois évalués (environ
6 jours), qui correspond à la dérive temporelle.

## 5. La préparation complète

In [8]:
inedits = ~evaluation["seller_id"].isin(entrainement["seller_id"])
encodeur_vendeur = prep.named_steps["encodage"].named_transformers_["vendeur"]

display(pd.DataFrame({
    "type": X_evaluation.dtypes.astype(str),
    "manquants (%)": 100 * X_evaluation.isna().mean(),
}).round(2))
pd.Series({
    "commandes d'avril de vendeurs inédits (%)": 100 * inedits.mean(),
    "délai encodé pour ces vendeurs (j)": X_evaluation.loc[inedits, "seller_id"].mean(),
    "délai moyen de l'entraînement (j)": encodeur_vendeur.target_mean_,
}).round(2)

,type,manquants (%)
distance_km,float64,0.51
meme_etat,int64,0.00
jour_semaine_achat,int32,0.00
prix_total,float64,0.00
fret_total,float64,0.00
poids_total_g,float64,0.00
volume_total_cm3,float64,0.00
n_articles,int64,0.00
n_echeances,float64,0.00
customer_state,float64,0.00


commandes d'avril de vendeurs inédits (%)     8.98
délai encodé pour ces vendeurs (j)           13.20
délai moyen de l'entraînement (j)            13.20
dtype: float64

La préparation transforme les 16 colonnes brutes en 14 variables numériques, prêtes pour le modèle :
- les catégories sont devenues des codes entiers. Une catégorie inconnue à l'entraînement devient une
  valeur manquante, évitant toute erreur dans l'API
- les 9 % de commandes d'avril venant de vendeurs inédits reçoivent la moyenne générale de
  l'entraînement, faute d'historique.

La même préparation, apprise sur les commandes livrées avant le 1ᵉʳ du mois, sera placée en tête de
chaque modèle futur. `predire_par_mois` la réentraîne automatiquement chaque mois.

## Synthèse

1. **Le modèle ne lit que 16 colonnes brutes**, toutes connues à l'achat. Les tests vérifient
   qu'aucune n'est postérieure à l'achat, et que modifier les colonnes postérieures ne change aucune
   variable.
2. **Une colonne suspecte a été vérifiée puis exclue**. Il s'agit du délai accordé au vendeur qui dépend du jour
   de validation du paiement (inconnu au moment de l'achat).
3. **14 variables** sont calculées commande par commande, avec le même résultat pour une commande
   seule ou pour tout l'historique (test). L'heure et le mois d'achat sont écartés.
4. **Le vendeur est encodé hors pli** : l'encodage naïf améliore l'entraînement mais dégrade chaque
   mois de validation.
5. **Les valeurs manquantes et les catégories inconnues** sont gérées sans erreur, condition
   nécessaire pour l'API.